In [64]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report


In [65]:
pd.set_option('display.max_columns',None)
url = 'https://raw.githubusercontent.com/ichiP245/my-next-soderling/refs/heads/main/Archivos/df_feng_done.csv'

df = pd.read_csv(url)
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y-%m-%d')

# df.loc[df['Location'] == 'Monte Carlo', 'Location'] = 'Monte Carlo, Monaco'

## Encoding de categóricas

Como el objetivo es que estos datos entren a un modelo de Machine Learning, le hacemos un encoding a las variables categóricas.

Para eso exploramos un poco esas variables. Sin embargo, para análisis exploratorio multivariable usaremos el df que acabamos de guardar, ya que tiene estas variables no encodeadas y eso nos va a facilitar el análisis para tal caso.

Para el nivel del torneo podemos hacer un OrdinalEncoder tranquilamente

In [ ]:
df['Series'].value_counts()

,count
Series,
Masters 1000,5884
Grand Slam,5459
ATP500,3954
ATP250,764


In [ ]:
from sklearn.preprocessing import OrdinalEncoder # Ordinal Encoder -> con valores de 0 a n cantidad de variables

oe = OrdinalEncoder(categories=[['ATP250', 'ATP500', 'Masters 1000', 'Grand Slam']])
series_oe = oe.fit_transform(df[['Series']])
series_oe = pd.DataFrame(series_oe, columns=['series_level_oe'])
df = pd.concat([df, series_oe], axis=1)

Para Round armamos varias One Hot, porque tenemos ['1st Round', '2nd Round', 'Quarterfinals', 'Semifinals', 'The Final', '3rd Round', '4th Round'] y no sirve hacer un solo OneHot o un LabelEncoder ni otro.

Entonces encodeamos las instancias que son mas importantes: si es o no una final, si es o no una semifinal (donde puede llegar alguno que hizo un batacazo y eso puede influenciar) y si es o no 1st round

In [ ]:
df['Round'].value_counts()

,count
Round,
1st Round,7252
2nd Round,4541
3rd Round,1768
Quarterfinals,1096
4th Round,583
Semifinals,548
The Final,273


In [ ]:
round_order = {
    '1st Round': 1, '2nd Round': 2, '3rd Round': 3,
    '4th Round': 4, 'Quarterfinals': 5, 'Semifinals': 6, 'The Final': 7
}

df['round_encoded'] = df['Round'].map(round_order)

"Best of" es numerica pero categorica. Le hacemos One Hot

In [ ]:
df['Best of'].value_counts()

,count
Best of,
3.0,10612
5.0,5449


In [ ]:
df['is_best_of_5'] = (df['Best of'] == 5).astype(int)

Para Surface = ['Hard', 'Clay', 'Grass'] usamos FrequencyEncoder

In [ ]:
df['Surface'].value_counts(normalize=True)

,proportion
Surface,
Hard,0.608617
Clay,0.293008
Grass,0.098375


In [ ]:
surface_fe = df['Surface'].value_counts(normalize=True) # Generamos un pd.Series con la proporcion de cada valor unicos
df['surface_fe'] = df['Surface'].map(surface_fe)  # Armamos una variable donde cada valor recibe la frecuencia de aparicion de ese valor unico

Para Court = ['Outdoor', 'Indoor'] usamos OneHotEncoder

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, drop='first')
court_encoded = ohe.fit_transform(df[['Court']])
court_encoded = pd.DataFrame(court_encoded, columns=ohe.get_feature_names_out()).rename(columns={'Court_Outdoor':'is_outdoor'})
df = pd.concat([df, court_encoded], axis=1)

Volvemos a guardar el df

In [ ]:
columns_drop = ['Series', 'Court', 'Surface', 'Round', 'Best of']

In [ ]:
df = df.drop(columns=columns_drop)

## Sigue preparación para ML

In [66]:
new_df = df.drop(columns=['Unnamed: 0', 'Location', 'Fecha', 'playerA', 'playerB',
                          'A1', 'B1', 'A2', 'B2', 'A3', 'B3', 'A4', 'B4', 'A5',
                          'B5', 'setsA', 'setsB', 'setsPartido'])

In [67]:
df_no_nulls = new_df.dropna()

In [68]:
X = df_no_nulls.drop(columns=['target'])
y = df_no_nulls['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)

Modelo 0: baseline

In [69]:
y_train_baseline = X_train[['rankA', 'rankB']].apply(lambda row: 1 if row['rankA'] > row['rankB'] else 0, axis=1)
y_test_baseline = X_test[['rankA', 'rankB']].apply(lambda row: 1 if row['rankA'] > row['rankB'] else 0, axis=1)

In [70]:
confusion_matrix(y_train, y_train_baseline)

array([[2130, 4113],
       [4161, 2008]])

In [71]:
print(classification_report(y_train, y_train_baseline))

              precision    recall  f1-score   support

           0       0.34      0.34      0.34      6243
           1       0.33      0.33      0.33      6169

    accuracy                           0.33     12412
   macro avg       0.33      0.33      0.33     12412
weighted avg       0.33      0.33      0.33     12412



In [51]:
confusion_matrix(y_test, y_test_baseline)

array([[ 530, 1063],
       [1026,  485]])

In [52]:
print(classification_report(y_test, y_test_baseline))

              precision    recall  f1-score   support

           0       0.34      0.33      0.34      1593
           1       0.31      0.32      0.32      1511

    accuracy                           0.33      3104
   macro avg       0.33      0.33      0.33      3104
weighted avg       0.33      0.33      0.33      3104



Modelo 1: RandomForest

In [72]:
rnd_clf = RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_split=4, random_state=42)
rnd_clf.fit(X_train, y_train)

RandomForestClassifier(max_depth=15, min_samples_split=4, n_estimators=500,
                       random_state=42)

In [73]:
y_pred_rf_train = rnd_clf.predict(X_train)
y_pred_rf_test = rnd_clf.predict(X_test)

In [74]:
confusion_matrix(y_train, y_pred_rf_train)

array([[5934,  309],
       [ 466, 5703]])

In [75]:
print(classification_report(y_train, y_pred_rf_train))

              precision    recall  f1-score   support

           0       0.93      0.95      0.94      6243
           1       0.95      0.92      0.94      6169

    accuracy                           0.94     12412
   macro avg       0.94      0.94      0.94     12412
weighted avg       0.94      0.94      0.94     12412



In [76]:
confusion_matrix(y_test, y_pred_rf_test)

array([[1128,  465],
       [ 412, 1099]])

In [77]:
print(classification_report(y_test, y_pred_rf_test))

              precision    recall  f1-score   support

           0       0.73      0.71      0.72      1593
           1       0.70      0.73      0.71      1511

    accuracy                           0.72      3104
   macro avg       0.72      0.72      0.72      3104
weighted avg       0.72      0.72      0.72      3104



In [78]:
importances = rnd_clf.feature_importances_
features = X.columns
feature_importances = pd.DataFrame({'feature': features, 'importance': importances}).sort_values('importance', ascending=False)
feature_importances

,feature,importance
15,ProbMaxA,0.051969
7,MaxB,0.046236
16,ProbMaxB,0.041461
14,ProbAvgB,0.039630
13,ProbAvgA,0.038809
6,MaxA,0.036475
21,logit_oddsA,0.035120
22,logit_oddsB,0.034089
20,AvgOddsDiff,0.029621
9,AvgB,0.027317


In [79]:
rnd_clf_2 = RandomForestClassifier(n_estimators=500, max_depth=10, min_samples_split=10, random_state=42)
rnd_clf_2.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, min_samples_split=10, n_estimators=500,
                       random_state=42)

In [80]:
y_pred_rf_2_train = rnd_clf.predict(X_train)
y_pred_rf_2_test = rnd_clf.predict(X_test)

In [81]:
print(confusion_matrix(y_test, y_pred_rf_2_test))
print(classification_report(y_test, y_pred_rf_2_test))

[[1128  465]
 [ 412 1099]]
              precision    recall  f1-score   support

           0       0.73      0.71      0.72      1593
           1       0.70      0.73      0.71      1511

    accuracy                           0.72      3104
   macro avg       0.72      0.72      0.72      3104
weighted avg       0.72      0.72      0.72      3104



In [86]:
pd.DataFrame(y_pred_rf_test)

,0
0,0
1,1
2,0
3,1
4,1
...,...
3099,1
3100,1
3101,0
3102,1


In [95]:
df_testeo = pd.concat([X_test.reset_index(drop=True), pd.DataFrame(y_test).reset_index(drop=True), pd.DataFrame(y_pred_rf_test, columns=['Pred'])], axis=1)

In [98]:
df_testeo

,rankA,rankB,PtsA,PtsB,B365A,B365B,MaxA,MaxB,AvgA,AvgB,B365BookmakersMargin,B365ProbA,B365ProbB,ProbAvgA,ProbAvgB,ProbMaxA,ProbMaxB,market_uncertainty,SpreadOddsA,SpreadOddsB,AvgOddsDiff,logit_oddsA,logit_oddsB,rankDiff,ptsDiff,winrate_5_A,winrate_10_A,wins_before_A,win_pct_before_A,sets_5d_A,sets_30d_A,partidos_365d_A,dias_ultimo_partido_A,racha_victorias_A,winrate_5_B,winrate_10_B,wins_before_B,win_pct_before_B,sets_5d_B,sets_30d_B,partidos_365d_B,dias_ultimo_partido_B,racha_victorias_B,diff_winrate_5,diff_winrate_10,diff_win_pct_before,diff_sets_5d,diff_sets_30d,diff_dias_ultimo_partido,diff_partidos_365d,h2h_matches_previous,series_level_oe,round_encoded,is_best_of_5,surface_fe,is_outdoor,target,Pred
0,145.0,6.0,395.0,4415.0,13.00,1.04,17.50,1.05,12.03,1.04,0.038462,0.074074,0.925926,0.079572,0.920428,0.056604,0.943396,0.420428,-0.022968,-0.022968,-0.840857,-2.448183,2.448183,139.0,-4020.0,0.2,0.3,12.0,0.400000,3.0,5.0,19,1.0,1,0.8,0.8,93.0,0.738095,0.0,10.0,49,7.0,0,-0.6,-0.5,-0.338095,3.0,-5.0,-6.0,-30,0,2.0,2,0,0.608617,1.0,0,0
1,40.0,219.0,1070.0,256.0,1.07,9.00,1.11,9.70,1.08,8.01,0.045691,0.893744,0.106256,0.881188,0.118812,0.897317,0.102683,0.381188,0.016129,0.016129,0.762376,2.003730,-2.003730,-179.0,814.0,0.6,0.7,118.0,0.655556,0.0,10.0,20,10.0,0,0.0,0.0,0.0,0.000000,0.0,0.0,0,1092.0,0,0.6,0.7,0.655556,0.0,10.0,-1082.0,20,0,3.0,1,1,0.098375,1.0,1,1
2,36.0,5.0,1230.0,5085.0,2.50,1.50,2.95,1.54,2.69,1.46,0.066667,0.375000,0.625000,0.351807,0.648193,0.342984,0.657016,0.148193,-0.008823,-0.008823,-0.296386,-0.611105,0.611105,31.0,-3855.0,0.4,0.5,88.0,0.628571,0.0,0.0,21,56.0,0,0.8,0.7,167.0,0.660079,0.0,18.0,53,11.0,0,-0.4,-0.2,-0.031508,0.0,-18.0,45.0,-32,1,1.0,1,0,0.608617,0.0,0,0
3,115.0,163.0,621.0,392.0,1.61,2.30,1.62,2.63,1.56,2.40,0.055901,0.588235,0.411765,0.606061,0.393939,0.618824,0.381176,0.106061,0.012763,0.012763,0.212121,0.430783,-0.430783,-48.0,229.0,0.6,0.5,3.0,0.500000,3.0,11.0,3,2.0,1,0.4,0.5,3.0,0.500000,3.0,7.0,4,2.0,1,0.2,0.0,0.000000,0.0,4.0,0.0,-1,0,0.0,2,0,0.608617,0.0,0,1
4,80.0,43.0,644.0,954.0,1.40,2.75,1.54,3.00,1.44,2.74,0.077922,0.662651,0.337349,0.655502,0.344498,0.660793,0.339207,0.155502,0.005291,0.005291,0.311005,0.643315,-0.643315,37.0,-310.0,0.4,0.4,4.0,0.400000,0.0,0.0,8,33.0,0,0.2,0.5,11.0,0.550000,0.0,2.0,19,7.0,0,0.2,-0.1,-0.150000,0.0,-2.0,26.0,-11,0,1.0,1,0,0.293008,1.0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3099,11.0,28.0,2900.0,1400.0,1.62,2.30,1.65,2.65,1.59,2.36,0.052067,0.586735,0.413265,0.597468,0.402532,0.616279,0.383721,0.097468,0.018811,0.018811,0.194937,0.394928,-0.394928,-17.0,1500.0,0.8,0.9,112.0,0.592593,7.0,25.0,49,1.0,3,0.8,0.8,43.0,0.443299,8.0,27.0,31,1.0,3,0.0,0.1,0.149294,-1.0,-2.0,0.0,18,1,1.0,6,0,0.608617,0.0,1,1
3100,8.0,6.0,4609.0,6595.0,2.00,1.80,2.07,1.91,1.97,1.82,0.055556,0.473684,0.526316,0.480211,0.519789,0.479899,0.520101,0.019789,-0.000312,-0.000312,-0.039578,-0.079197,0.079197,2.0,-1986.0,0.8,0.8,86.0,0.623188,7.0,21.0,33,1.0,3,0.8,0.8,96.0,0.640000,8.0,27.0,31,1.0,3,0.0,0.0,-0.016812,-1.0,-6.0,0.0,2,3,1.0,6,0,0.608617,0.0,1,1
3101,19.0,12.0,2346.0,2830.0,2.75,1.44,3.06,1.57,2.72,1.46,0.058081,0.343675,0.656325,0.349282,0.650718,0.339093,0.660907,0.150718,-0.010189,-0.010189,-0.301435,-0.622195,0.622195,7.0,-484.0,0.4,0.4,34.0,0.539683,6.0,6.0,19,2.0,2,0.6,0.7,64.0,0.576577,8.0,8.0,17,2.0,2,-0.2,-0.3,-0.036894,-2.0,-2.0,0.0,2,3,3.0,3,1,0.608617,1.0,1,0
3102,3.0,24.0,6500.0,1970.0,1.13,6.00,1.16,7.04,1.14,5.90,0.051622,0.841515,0.158485,0.838068,0.161932,0.858537,0.141463,0.338068,0.020468,0.020468,0.676136,1.643924,-1.643924,-21.0,4530.0,0.8,0.7,381.0,0.714822,5.0,23.0,72,1.0,2,0.8,0.9,37.0,0.536232,6.0,25.0,46,1.0,2,0.0,-0.2,0.178590,-1.0,-2.0,0.0,26,1,1.0,5,0,0.098375,1.0,1,1


In [101]:
sum(df_testeo.apply(lambda row: row['B365A']*1 - 1 if row['target'] == row['Pred'] else -1, axis=1))

4025.036